# V11 Improved Inference - Target: >0.562 LB

## OOM Root Cause & Fix
- `SurfaceRefinementBlock.edge_conv` (Conv3d 64→64, k=3) at 192³ needs **22.78 GiB**
- Even batch_size=1 on a single T4 (14.56 GB) → OOM
- **Fix: Use 128³ patches** — memory drops to ~6.5 GB (fits any T4)
- Model is fully convolutional — works with any input size
- Higher overlap (0.75) compensates for smaller patches

## Deep Supervision Ensemble (NEW)
- ds_heads are trained 1x1 Conv3d layers at intermediate decoder levels
- At inference: predict from ALL decoder levels, upsample, weighted average
- Weights: main=0.70, ds0=0.15, ds1=0.10, ds2=0.05
- **FREE multi-scale ensemble** — no extra training needed!

## Improvements
| Change | Previous (0.562) | This Version |
|--------|-----------------|-------------|
| Patch size | 192³ (OOM risk) | **128³** (safe on any T4) |
| Overlap | 0.5 | **0.75** (very smooth prob map) |
| TTA | flip only (4 preds) | **flip (4 preds)** |
| Post-processing | threshold 0.5 | **Hysteresis (0.45/0.25)** |
| Prob smoothing | None | **Gaussian sigma=0.5** |
| Deep supervision | off at inference | **Ensemble (4 scales)** |

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import warnings
import zipfile
from pathlib import Path
from typing import List, Tuple
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.ndimage import (
    binary_fill_holes, distance_transform_edt, gaussian_filter,
    label, generate_binary_structure, binary_dilation,
    binary_closing, binary_opening, binary_erosion
)
from skimage.morphology import skeletonize

warnings.filterwarnings('ignore')

# =============================================================================
# GPU SETUP
# =============================================================================
N_GPUS = torch.cuda.device_count()
print(f"Available GPUs: {N_GPUS}")
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} ({props.total_memory / 1e9:.1f} GB)")

@dataclass
class Config:
    # Paths
    TEST_ROOT: Path = Path("/kaggle/input/vesuvius-challenge-surface-detection/test_images")
    CHECKPOINT_PATH: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/5/checkpoints_v11/fold_0/best_model.pth")
    OUTPUT_DIR: Path = Path("/kaggle/working")
    
    # Model (must match training)
    FEATURES: List[int] = field(default_factory=lambda: [32, 64, 128, 256, 320, 320])
    N_BLOCKS: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 6, 6])
    
    # =========================================================================
    # INFERENCE SETTINGS - OOM FIX + IMPROVEMENTS
    # =========================================================================
    # 192³ patch → SurfaceRefinementBlock needs 22.78 GB → OOM on T4 (14.56 GB)
    # 128³ patch → SurfaceRefinementBlock needs ~6.5 GB → SAFE on any T4
    # Memory scales as O(n³): (128/192)³ = 0.296x memory
    # Higher overlap (0.75) compensates for smaller context window
    # =========================================================================
    INFER_PATCH_SIZE: Tuple[int, int, int] = (128, 128, 128)  # REDUCED from 192
    OVERLAP: float = 0.75        # INCREASED from 0.5 (compensates for smaller patches)
    TTA_LEVEL: str = "flip"      # 4 predictions (orig + 3 flips)
    USE_FLOAT16: bool = True
    
    # With 128³ patches, batch_size=2 is safe on T4
    USE_MULTI_GPU: bool = True
    BATCH_SIZE: int = 2           # Safe with 128³ (~6.5 GB per patch)
    
    # Post-processing - IMPROVED
    THRESHOLD_HIGH: float = 0.45  # Hysteresis seed threshold
    THRESHOLD_LOW: float = 0.25   # Hysteresis growth threshold
    MIN_COMPONENT_SIZE: int = 50
    PROB_SMOOTH_SIGMA: float = 0.5
    
    DEVICE: str = "cuda"

cfg = Config()

print(f"\nConfiguration:")
print(f"  Patch: {cfg.INFER_PATCH_SIZE} (reduced from 192³ to avoid OOM)")
print(f"  Overlap: {cfg.OVERLAP} (increased from 0.5 to compensate)")
print(f"  TTA: {cfg.TTA_LEVEL}")
print(f"  Batch size: {cfg.BATCH_SIZE}")
print(f"  Hysteresis: high={cfg.THRESHOLD_HIGH}, low={cfg.THRESHOLD_LOW}")
print(f"  Prob smoothing sigma: {cfg.PROB_SMOOTH_SIGMA}")

for i in range(N_GPUS):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
gc.collect()

In [ ]:
# =============================================================================
# MODEL ARCHITECTURE (MUST MATCH TRAINING EXACTLY)
# =============================================================================
# DEEP SUPERVISION ENSEMBLE FOR INFERENCE:
#   During training, ds_heads only run when self.training == True.
#   But these heads ARE trained 1x1 Conv3d layers that predict segmentation
#   from intermediate decoder features at different scales.
#   
#   At inference, we can use them for a FREE multi-scale ensemble:
#   - ds_heads[0]: predicts from decoder level 1 (coarsest, 320ch)
#   - ds_heads[1]: predicts from decoder level 2 (256ch)  
#   - ds_heads[2]: predicts from decoder level 3 (128ch)
#   - final: predicts from decoder level 5 (finest, 32ch)
#   
#   All upsampled to full resolution and averaged with learned weights.
#   This acts like a built-in multi-scale ensemble at zero extra training cost.
# =============================================================================

def get_num_groups(channels, max_groups=32):
    for g in [max_groups, 16, 8, 4, 2, 1]:
        if channels % g == 0:
            return g
    return 1


class HybridConv3d(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid_ch = out_ch // 2
        self.conv_xy = nn.Conv3d(in_ch, mid_ch, kernel_size=(1, 3, 3), padding=(0, 1, 1), bias=False)
        self.conv_z = nn.Conv3d(in_ch, out_ch - mid_ch, kernel_size=(3, 1, 1), padding=(1, 0, 0), bias=False)
        self.norm = nn.GroupNorm(get_num_groups(out_ch), out_ch)
        self.act = nn.LeakyReLU(0.01, inplace=True)
    def forward(self, x):
        return self.act(self.norm(torch.cat([self.conv_xy(x), self.conv_z(x)], dim=1)))


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_hybrid=False):
        super().__init__()
        if use_hybrid:
            self.conv = HybridConv3d(in_ch, out_ch)
        else:
            self.conv = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.GroupNorm(get_num_groups(out_ch), out_ch),
                nn.LeakyReLU(0.01, inplace=True),
            )
    def forward(self, x): return self.conv(x)


class MultiScaleResBlock(nn.Module):
    def __init__(self, channels, scales=2, use_hybrid=False):
        super().__init__()
        self.scales = scales
        self.width = channels // scales
        self.convs = nn.ModuleList([ConvBlock(self.width, self.width, use_hybrid=use_hybrid) for _ in range(scales - 1)])
        self.norm = nn.GroupNorm(get_num_groups(channels), channels)
    def forward(self, x):
        splits = torch.chunk(x, self.scales, dim=1)
        outputs = [splits[0]]
        for i, conv in enumerate(self.convs):
            out = conv(splits[i+1] + outputs[-1]) if i > 0 else conv(splits[i+1])
            outputs.append(out)
        return x + self.norm(torch.cat(outputs, dim=1))


class AttentionBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Linear(channels, max(channels // reduction, 8))
        self.fc2 = nn.Linear(max(channels // reduction, 8), channels)
        self.conv_spatial = nn.Conv3d(2, 1, kernel_size=7, padding=3, bias=False)
    def forward(self, x):
        b, c = x.shape[:2]
        ca = torch.sigmoid(self.fc2(F.relu(self.fc1(self.gap(x).view(b, c))))).view(b, c, 1, 1, 1)
        x_ca = x * ca
        sa = torch.sigmoid(self.conv_spatial(torch.cat([x_ca.mean(1, keepdim=True), x_ca.max(1, keepdim=True)[0]], 1)))
        return x_ca * sa


class SurfaceRefinementBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.edge_conv = nn.Conv3d(in_ch, in_ch, 3, padding=1, bias=False)
        self.refine = nn.Sequential(
            nn.Conv3d(in_ch * 2, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )
    def forward(self, x):
        edge = torch.abs(self.edge_conv(x))
        combined = torch.cat([x, edge], dim=1)
        del edge
        return self.refine(combined)


class TopoPreservingUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None,
                 use_attention=True, use_hybrid_conv=True, use_surface_refinement=True,
                 use_deep_supervision=True):
        super().__init__()
        features = features or [32, 64, 128, 256, 320, 320]
        n_blocks = n_blocks or [1, 2, 3, 4, 6, 6]
        self.features = features
        self.use_deep_supervision = use_deep_supervision
        
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i-1]
            layers = [ConvBlock(in_channels, feat, use_hybrid=use_hybrid_conv and i > 0)]
            layers.extend([MultiScaleResBlock(feat, scales=2, use_hybrid=use_hybrid_conv) for _ in range(nb)])
            self.encoders.append(nn.Sequential(*layers))
            self.attentions.append(AttentionBlock(feat) if use_attention and i >= 2 else nn.Identity())
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, stride=2))
        
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features)-2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i+1], features[i], 2, stride=2))
            if use_surface_refinement and i == 0:
                self.dec_convs.append(SurfaceRefinementBlock(features[i]*2, features[i]))
            else:
                self.dec_convs.append(nn.Sequential(
                    ConvBlock(features[i]*2, features[i], use_hybrid=use_hybrid_conv),
                    MultiScaleResBlock(features[i], scales=2, use_hybrid=use_hybrid_conv),
                ))
        
        self.final = nn.Conv3d(features[0], out_ch, 1)
        
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList([
                nn.Conv3d(features[-(i+2)], out_ch, 1) for i in range(min(3, len(features)-1))
            ])
    
    def forward(self, x):
        """
        MODIFIED FOR INFERENCE: Deep supervision ensemble.
        
        During training (self.training=True):
            - Returns {'output': main_pred, 'deep': [ds_pred1, ds_pred2, ds_pred3]}
            - Same as original
            
        During inference (self.training=False):
            - Uses ds_heads to predict at intermediate decoder levels
            - Upsamples all to full resolution
            - Returns weighted average: main=0.70, ds0=0.15, ds1=0.10, ds2=0.05
            - This is a FREE multi-scale ensemble (weights already trained!)
        """
        target_shape = x.shape[2:]  # Save input spatial dims for DS upsampling
        
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = att(enc(x))
            enc_features.append(x)
            if i < len(self.pools): x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        ds_outputs = []
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i+1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
            
            # Collect DS outputs for BOTH training and inference
            if self.use_deep_supervision and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](x))
            
            enc_features[i+1] = None  # Free memory
        
        out = self.final(x)
        
        if self.training:
            # Training: return dict for loss computation (same as original)
            return {'output': out, 'deep': ds_outputs} if self.use_deep_supervision else out
        else:
            # Inference: deep supervision ensemble
            if self.use_deep_supervision and len(ds_outputs) > 0:
                # Ensemble weights: main output gets most weight, DS heads add refinement
                # main=0.70, ds0=0.15, ds1=0.10, ds2=0.05
                ds_weights = [0.15, 0.10, 0.05]
                main_weight = 0.70
                
                ensemble = out * main_weight
                for i, ds_out in enumerate(ds_outputs):
                    if i >= len(ds_weights):
                        break
                    # Upsample DS output to full resolution
                    if ds_out.shape[2:] != target_shape:
                        ds_up = F.interpolate(ds_out, size=target_shape, mode='trilinear', align_corners=False)
                    else:
                        ds_up = ds_out
                    ensemble = ensemble + ds_weights[i] * ds_up
                    del ds_up
                
                return ensemble
            else:
                return out

print("Model architecture defined (WITH deep supervision ensemble for inference)")
print("  Training: returns {'output': pred, 'deep': [ds1, ds2, ds3]}")
print("  Inference: returns weighted ensemble (main=0.70, ds0=0.15, ds1=0.10, ds2=0.05)")
print("  → Free multi-scale ensemble using trained ds_heads!")

In [ ]:
# =============================================================================
# INFERENCE ENGINE
# =============================================================================
# 128³ patches with overlap 0.75 → many more patches but each fits in memory
# Gaussian weighting ensures smooth blending at patch boundaries
# =============================================================================

def robust_zscore_normalize(img, lower_percentile=0.5, upper_percentile=99.5):
    """MUST MATCH TRAINING EXACTLY - percentile clipping + Z-score."""
    p_low = np.percentile(img, lower_percentile)
    p_high = np.percentile(img, upper_percentile)
    img_clipped = np.clip(img, p_low, p_high)
    mean = img_clipped.mean()
    std = img_clipped.std()
    return ((img_clipped - mean) / (std + 1e-8)).astype(np.float32)


def create_gaussian_weight(patch_size, sigma=0.125):
    d, h, w = patch_size
    gz = np.exp(-0.5 * ((np.arange(d) - d/2) / (d * sigma)) ** 2)
    gy = np.exp(-0.5 * ((np.arange(h) - h/2) / (h * sigma)) ** 2)
    gx = np.exp(-0.5 * ((np.arange(w) - w/2) / (w * sigma)) ** 2)
    return (gz[:, None, None] * gy[None, :, None] * gx[None, None, :]).astype(np.float32)


def get_patch_positions(volume_shape, patch_size, overlap=0.5):
    D, H, W = volume_shape
    pd, ph, pw = patch_size
    sd, sh, sw = int(pd*(1-overlap)), int(ph*(1-overlap)), int(pw*(1-overlap))
    sd, sh, sw = max(1, sd), max(1, sh), max(1, sw)
    
    z_pos = list(range(0, max(1, D-pd+1), sd))
    if len(z_pos) == 0 or z_pos[-1] + pd < D: z_pos.append(max(0, D - pd))
    y_pos = list(range(0, max(1, H-ph+1), sh))
    if len(y_pos) == 0 or y_pos[-1] + ph < H: y_pos.append(max(0, H - ph))
    x_pos = list(range(0, max(1, W-pw+1), sw))
    if len(x_pos) == 0 or x_pos[-1] + pw < W: x_pos.append(max(0, W - pw))
    
    # Deduplicate
    z_pos = sorted(set(z_pos))
    y_pos = sorted(set(y_pos))
    x_pos = sorted(set(x_pos))
    
    positions = []
    for z in z_pos:
        for y in y_pos:
            for x in x_pos:
                positions.append((z, y, x))
    return positions


@torch.no_grad()
def sliding_window_inference(model, volume, patch_size, overlap=0.75, batch_size=1):
    """
    Sliding window inference with 128³ patches.
    Memory: ~6.5 GB per patch (safe for T4 14.56 GB).
    """
    model.eval()
    
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    
    # Pad if needed
    pad_d, pad_h, pad_w = max(0, pd-D), max(0, ph-H), max(0, pw-W)
    if pad_d > 0 or pad_h > 0 or pad_w > 0:
        volume = np.pad(volume, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='reflect')
        D, H, W = volume.shape
    
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    weight_sum = np.zeros((D, H, W), dtype=np.float32)
    gauss = create_gaussian_weight(patch_size)
    
    positions = get_patch_positions((D, H, W), patch_size, overlap)
    vol_norm = robust_zscore_normalize(volume)
    
    print(f"  Patches: {len(positions)}, Batch: {batch_size}, Overlap: {overlap}")
    
    for batch_start in tqdm(range(0, len(positions), batch_size), desc="Inference"):
        batch_end = min(batch_start + batch_size, len(positions))
        batch_positions = positions[batch_start:batch_end]
        
        patches = [vol_norm[z:z+pd, y:y+ph, x:x+pw] for (z, y, x) in batch_positions]
        batch_tensor = torch.from_numpy(np.stack(patches)[:, None]).cuda().half()
        
        with torch.cuda.amp.autocast(dtype=torch.float16):
            batch_pred = torch.sigmoid(model(batch_tensor))
        batch_pred = batch_pred.squeeze(1).float().cpu().numpy()
        
        for i, (z, y, x) in enumerate(batch_positions):
            pred_sum[z:z+pd, y:y+ph, x:x+pw] += batch_pred[i] * gauss
            weight_sum[z:z+pd, y:y+ph, x:x+pw] += gauss
        
        del batch_tensor, batch_pred, patches
        if (batch_start // max(batch_size, 1)) % 20 == 0:
            torch.cuda.empty_cache()
    
    pred = pred_sum / np.maximum(weight_sum, 1e-8)
    
    if pad_d > 0: pred = pred[:-pad_d]
    if pad_h > 0: pred = pred[:, :-pad_h]
    if pad_w > 0: pred = pred[:, :, :-pad_w]
    
    return pred


def _clear_gpu():
    gc.collect()
    for i in range(N_GPUS):
        with torch.cuda.device(i):
            torch.cuda.empty_cache()


@torch.no_grad()
def inference_with_tta(model, volume, patch_size, overlap=0.75, batch_size=1, tta='flip'):
    """
    TTA: original + 3 axis flips = 4 predictions.
    """
    print("  TTA: Original")
    pred = sliding_window_inference(model, volume, patch_size, overlap, batch_size)
    
    if tta == 'none':
        return pred
    
    preds = [pred]
    del pred
    _clear_gpu()
    
    # Flip augmentations (3 axes)
    if tta in ['flip', 'full']:
        for axis in [0, 1, 2]:
            print(f"  TTA: Flip axis {axis}")
            vol_flip = np.flip(volume, axis).copy()
            pred_flip = sliding_window_inference(model, vol_flip, patch_size, overlap, batch_size)
            preds.append(np.flip(pred_flip, axis).copy())
            del vol_flip, pred_flip
            _clear_gpu()
    
    # Rotation augmentation (rot90 in YZ plane)
    if tta == 'full':
        print("  TTA: Rot90 (axes 1,2)")
        vol_rot = np.rot90(volume, k=1, axes=(1, 2)).copy()
        pred_rot = sliding_window_inference(model, vol_rot, patch_size, overlap, batch_size)
        preds.append(np.rot90(pred_rot, k=-1, axes=(1, 2)).copy())
        del vol_rot, pred_rot
        _clear_gpu()
    
    print(f"  TTA: Averaging {len(preds)} predictions")
    result = np.mean(preds, axis=0)
    del preds
    gc.collect()
    return result

print("Inference engine ready")
print(f"  Patch: {cfg.INFER_PATCH_SIZE} (128³ = safe on any T4)")
print(f"  Overlap: {cfg.OVERLAP}")
print(f"  TTA: {cfg.TTA_LEVEL}")
print(f"  Batch: {cfg.BATCH_SIZE}")

In [ ]:
# =============================================================================
# POST-PROCESSING - IMPROVED
# =============================================================================
# Learnings from all submissions:
#   - Skeleton-guided (threshold 0.70) = 0.471 (too aggressive)
#   - Simple threshold 0.50 + slicewise morph = 0.562 (best so far)
#   - Simple threshold 0.80 = ~0.538 (too conservative)
#
# Improvement: Hysteresis thresholding
#   - V11 model mean prob is only ~0.10 → threshold 0.5 cuts border regions
#   - Seeds from high confidence (>0.45) + grow into low (>0.25)
#   - Captures more valid structure without adding disconnected noise
# =============================================================================

def count_components(mask):
    struct = generate_binary_structure(3, 3)
    _, n = label(mask, structure=struct)
    return n


def remove_small_components(mask, min_size=50):
    struct = generate_binary_structure(3, 3)
    labeled, n = label(mask, structure=struct)
    if n == 0:
        return mask
    sizes = np.bincount(labeled.ravel())
    small = sizes < min_size
    small[0] = False
    result = mask.copy()
    result[small[labeled]] = 0
    return result


def hysteresis_threshold(prob_map, high=0.45, low=0.25):
    """
    Hysteresis thresholding:
    1. Seeds from high-confidence voxels (prob > high)
    2. Candidates from low-confidence voxels (prob > low)
    3. Keep only candidates connected to seeds (26-connectivity)
    """
    seeds = (prob_map > high).astype(np.uint8)
    candidates = (prob_map > low).astype(np.uint8)
    
    print(f"  Hysteresis: seeds (>{high}): FG={100*seeds.mean():.2f}%, "
          f"candidates (>{low}): FG={100*candidates.mean():.2f}%")
    
    if seeds.sum() == 0:
        print("  WARNING: No seeds! Falling back to low threshold.")
        return candidates
    
    struct = generate_binary_structure(3, 3)
    labeled_candidates, n_candidates = label(candidates, structure=struct)
    
    seed_labels = set(np.unique(labeled_candidates[seeds > 0]))
    seed_labels.discard(0)
    
    result = np.zeros_like(candidates)
    for lbl in seed_labels:
        result[labeled_candidates == lbl] = 1
    
    print(f"  Hysteresis result: FG={100*result.mean():.2f}%, "
          f"components: {n_candidates} -> {len(seed_labels)}")
    
    return result


def topology_safe_operation(mask, operation_func, name="op"):
    n_before = count_components(mask)
    result = operation_func(mask)
    n_after = count_components(result)
    if n_after < n_before:
        print(f"    [REVERT] {name}: would merge {n_before}->{n_after} components")
        return mask
    return result


def slicewise_hole_fill(mask):
    filled = mask.copy()
    for i in range(mask.shape[0]):
        filled[i] = binary_fill_holes(filled[i])
    for i in range(mask.shape[1]):
        filled[:, i, :] = binary_fill_holes(filled[:, i, :])
    for i in range(mask.shape[2]):
        filled[:, :, i] = binary_fill_holes(filled[:, :, i])
    return filled


def slicewise_morphology(mask, operation='close', iterations=1):
    result = mask.copy()
    struct_2d = generate_binary_structure(2, 1)
    op_func = binary_closing if operation == 'close' else binary_opening
    for axis in range(3):
        for i in range(mask.shape[axis]):
            if axis == 0:
                result[i] = op_func(result[i], structure=struct_2d, iterations=iterations)
            elif axis == 1:
                result[:, i, :] = op_func(result[:, i, :], structure=struct_2d, iterations=iterations)
            else:
                result[:, :, i] = op_func(result[:, :, i], structure=struct_2d, iterations=iterations)
    return result


def improved_postprocess(pred_prob, 
                         threshold_high=0.45,
                         threshold_low=0.25,
                         min_component_size=50,
                         smooth_sigma=0.5,
                         verbose=True):
    if verbose:
        print("Post-processing (IMPROVED):")
        print(f"  Input: min={pred_prob.min():.3f}, max={pred_prob.max():.3f}, mean={pred_prob.mean():.3f}")
    
    # Step 1: Light Gaussian smoothing
    if smooth_sigma > 0:
        pred_smooth = gaussian_filter(pred_prob, sigma=smooth_sigma)
        if verbose:
            print(f"  1. Gaussian smooth (sigma={smooth_sigma}): "
                  f"mean {pred_prob.mean():.3f} -> {pred_smooth.mean():.3f}")
        pred_prob = pred_smooth
    
    # Step 2: Hysteresis thresholding
    mask = hysteresis_threshold(pred_prob, high=threshold_high, low=threshold_low)
    if verbose:
        print(f"  2. After hysteresis: {mask.sum():,} voxels, FG={100*mask.mean():.2f}%")
    
    if mask.sum() == 0:
        if verbose: print("  WARNING: Empty mask!")
        return mask.astype(np.uint8)
    
    # Step 3: Remove small components
    n_before = count_components(mask)
    mask = remove_small_components(mask, min_component_size)
    n_after = count_components(mask)
    if verbose:
        print(f"  3. Remove small (<{min_component_size}): {n_before}->{n_after} components")
    
    # Step 4: Slicewise closing (topology-safe)
    mask = topology_safe_operation(
        mask,
        lambda m: slicewise_morphology(m, 'close', iterations=1),
        "slicewise_close"
    )
    if verbose:
        print(f"  4. Slicewise closing: FG={100*mask.mean():.2f}%")
    
    # Step 5: Slicewise hole filling (topology-safe)
    mask = topology_safe_operation(mask, slicewise_hole_fill, "slicewise_hole_fill")
    if verbose:
        print(f"  5. Slicewise hole fill: FG={100*mask.mean():.2f}%")
    
    # Step 6: Slicewise opening (topology-safe)
    mask = topology_safe_operation(
        mask,
        lambda m: slicewise_morphology(m, 'open', iterations=1),
        "slicewise_open"
    )
    if verbose:
        print(f"  6. Slicewise opening: FG={100*mask.mean():.2f}%")
    
    # Step 7: Final cleanup
    mask = remove_small_components(mask, min_component_size)
    n_final = count_components(mask)
    if verbose:
        print(f"  Final: {n_final} components, {mask.sum():,} voxels, FG={100*mask.mean():.2f}%")
    
    return mask.astype(np.uint8)


print("Post-processing ready")
print(f"  Hysteresis: high={cfg.THRESHOLD_HIGH}, low={cfg.THRESHOLD_LOW}")
print(f"  Smoothing: sigma={cfg.PROB_SMOOTH_SIGMA}")

In [ ]:
# =============================================================================
# LOAD MODEL
# =============================================================================

for i in range(N_GPUS):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
gc.collect()

print(f"Loading: {cfg.CHECKPOINT_PATH}")

model = TopoPreservingUNet3D(
    features=cfg.FEATURES,
    n_blocks=cfg.N_BLOCKS,
    use_attention=True,
    use_hybrid_conv=True,
    use_surface_refinement=True,
    use_deep_supervision=True,
).cuda()

ckpt = torch.load(cfg.CHECKPOINT_PATH, map_location='cuda:0', weights_only=False)
state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}

missing, unexpected = model.load_state_dict(state, strict=False)
if missing:
    print(f"  WARNING - Missing keys ({len(missing)})")
    for k in missing[:5]: print(f"    {k}")
if unexpected:
    print(f"  WARNING - Unexpected keys ({len(unexpected)})")
    for k in unexpected[:5]: print(f"    {k}")
if not missing and not unexpected:
    print(f"  ALL weights loaded perfectly!")

print(f"  Epoch: {ckpt.get('epoch', '?')}, Best Dice: {ckpt.get('best_dice', '?'):.4f}")

# Use DataParallel if 2 GPUs available (safe with 128³ patches)
if N_GPUS > 1 and cfg.USE_MULTI_GPU:
    print(f"\n>>> DataParallel across {N_GPUS} GPUs (safe with 128³ patches)")
    model = nn.DataParallel(model, device_ids=list(range(N_GPUS)))
else:
    print(f"\n>>> Single GPU mode")

model.half()
model.eval()

print(f"\nGPU Memory:")
for i in range(N_GPUS):
    mem = torch.cuda.memory_allocated(i) / 1e9
    print(f"  GPU {i}: {mem:.2f} GB")
print("Model ready!")

In [ ]:
# =============================================================================
# RUN INFERENCE + POST-PROCESSING
# =============================================================================

test_files = sorted(cfg.TEST_ROOT.glob("*.tif")) + sorted(cfg.TEST_ROOT.glob("*.tiff"))
print(f"Found {len(test_files)} test volumes")

mask_dir = cfg.OUTPUT_DIR / "masks"
mask_dir.mkdir(exist_ok=True)

for vol_path in test_files:
    vol_id = vol_path.stem
    print(f"\n{'='*70}")
    print(f"Processing: {vol_id}")
    print(f"{'='*70}")
    
    _clear_gpu()
    
    volume = tifffile.imread(str(vol_path)).astype(np.float32)
    original_shape = volume.shape
    print(f"Shape: {original_shape}")
    
    # Inference
    print(f"Running inference (patch={cfg.INFER_PATCH_SIZE}, overlap={cfg.OVERLAP}, TTA={cfg.TTA_LEVEL})...")
    pred_prob = inference_with_tta(
        model, volume, cfg.INFER_PATCH_SIZE, cfg.OVERLAP,
        cfg.BATCH_SIZE, cfg.TTA_LEVEL
    )
    
    del volume
    _clear_gpu()
    
    # Post-processing
    pred_mask = improved_postprocess(
        pred_prob,
        threshold_high=cfg.THRESHOLD_HIGH,
        threshold_low=cfg.THRESHOLD_LOW,
        min_component_size=cfg.MIN_COMPONENT_SIZE,
        smooth_sigma=cfg.PROB_SMOOTH_SIGMA,
        verbose=True,
    )
    
    assert pred_mask.shape == original_shape, f"Shape mismatch: {pred_mask.shape} vs {original_shape}"
    
    # Save
    mask_path = mask_dir / f"{vol_id}.tif"
    tifffile.imwrite(str(mask_path), pred_mask.astype(np.uint8), compression=None)
    actual_size = mask_path.stat().st_size / 1e6
    print(f"  Saved: {mask_path.name} ({actual_size:.2f} MB)")
    
    del pred_prob, pred_mask
    _clear_gpu()

print(f"\n{'='*70}")
print("INFERENCE COMPLETE")
print(f"{'='*70}")

In [ ]:
# =============================================================================
# CREATE SUBMISSION ZIP
# =============================================================================

submission_zip = cfg.OUTPUT_DIR / "submission.zip"
with zipfile.ZipFile(submission_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for tif_path in sorted(mask_dir.glob("*.tif")):
        zf.write(tif_path, tif_path.name)
        print(f"  Added: {tif_path.name}")

print(f"\nSubmission: {submission_zip}")
with zipfile.ZipFile(submission_zip, 'r') as zf:
    for info in zf.infolist():
        print(f"  {info.filename}: {info.file_size / 1e6:.2f} MB")

print(f"\nReady to submit!")